In [1]:
import numpy as np
import tifffile as tiff
from matplotlib import pyplot as plt
import cv2 as cv
import scipy as sp
import trad_cv_method as tradcv
from scipy.ndimage import shift
from skimage.registration import phase_cross_correlation
from skimage.transform import AffineTransform, warp
import matplotlib as mpl
from skimage import data, img_as_float
from skimage import exposure
import skimage
from PIL import Image
import imageio.v2 as imageio
import os

## **SAVE FRAME AVERAGED MICROSCOPY VIDEOS**

Currently, the DeepCAD denoising model we trained needs 3 frame averaged data. And, before we develop a method to quantify Ca2+ signals from that data, we are performing 12 frame averaging and then using the old analysis methods. Keep in mind this is not a rolling average; the resultant video is `len(raw video) / n` frames long, where `n` is the number of frames for averaging.

In [3]:
def frameAvg(video, avg_size):
    averaged_vid = np.zeros((len(video)//avg_size, len(video[0,:]), len(video[0,0,:])))
    for i in range(0, len(video)-len(video)%avg_size, avg_size):
        averaged_frame = np.mean(video[i:i+avg_size], axis=0)
        averaged_vid[i//avg_size] = averaged_frame
    return averaged_vid

Set `parent_directory` and `directories_list` in the cell below.
* `raw_microscope_recordings_dir` is the folder which contains all raw microscopy files for an animal.
* `directories_list` is a list of folders containing specific recordings of interest
  
**Example:** 
* `raw_microscope_recordings_dir` = `'/Volumes/NVR/Max/Anes Scans/6xFAD ctrl 65 F 010225 032725'`
* `directories_list` = <br>
  `['6xFAD ctrl F 65 FOV1 postbarium 2D', '6xFAD ctrl F 65 FOV1 prebarium 2D',` <br>
   ` '6xFAD ctrl F 65 FOV2 postbarium 2D', '6xFAD ctrl F 65 FOV2 prebarium 2D']`

In this example, you will frame average **pre and post barium** recordings from **FOV1** and **FOV2** of **6xFAD ctrl 65 F**


In [12]:

raw_microscope_recordings_dir = '/Volumes/NVR_Local_Server/Max/Anes Scans/20250512 6xFAD 91 M anes scan' # folder containing all recordings/files for a specific animal
animal_id = raw_microscope_recordings_dir.split('/')[-1]
print(animal_id)

# folder names containing individual recordings you want
directories_list = ["20250512 6xFAD 91 tg+ FOV1 2D-000", "20250512 6xFAD 91 tg+ FOV1 post barium-000",
                    "20250512 6xFAD 91 tg+ FOV2 2D-000", "20250512 6xFAD 91 tg+ FOV2 post barium-000"
                   ] 

paths_dict = {}

for subfolder in directories_list:
    directory = os.path.join(raw_microscope_recordings_dir, subfolder)
    
    for file in os.listdir(directory):
        if ".tif" in file:
            file_no_suffix = file[:file.find('_00000')]
            if file_no_suffix not in paths_dict:
                paths_dict[file_no_suffix] = [os.path.join(directory, file)]
            else:
                paths_dict[file_no_suffix].append(os.path.join(directory, file))

20250512 6xFAD 91 M anes scan


Set `project_data_dir` to the main folder where all data (frame averaged tiffs, ST Maps, etc...) for each project will be stored. The code assumes that inside the `project_data_dir` folder, there will be a "12favg_combined" folder which will contain folders containing recordings from each animal. For more details, see documentation on folder structure in the instructions document.

Set `n_avg` to the number of frames you want to average by. 

The code will automatically stitch broken up .tiffs of the same recording together. If you are saving .tiffs for 12 frame averaged ST Map generation, make sure that the file structure is according to the assumed directory structure for 12 frame averaged data. Note you will have to manually create a Plasma subfolder afterwards for the Plasma channel tiffs.

In [19]:
# name of folder where you want to save the frame averaged videos
# make sure you create the folder in the drive first before you run
n_avg = 12 # MAKE SURE THIS IS SET TO THE CORRECT NUMBER OF FRAMES

project_data_dir = "/Volumes/NVR_Local_Server/Joshua/12favg_data_5xfad_barium"
averaged_vid_directory = os.path.join(project_data_dir, "12favg_combined/", animal_id)

if not os.path.exists(averaged_vid_directory):
    os.mkdir(os.path.join(project_data_dir, "12favg_combined/", animal_id))
if not os.path.exists(os.path.join(averaged_vid_directory, 'Plasma')):
    os.mkdir(os.path.join(averaged_vid_directory, 'Plasma'))

for recording in paths_dict:
    filepath_list = paths_dict[recording]
    filepath_list.sort()
    vids_list = []
    for path in filepath_list:
        video = tiff.imread(path)
        video_avg = frameAvg(video, n_avg)
        vids_list.append(video_avg)
    combined_vid = np.concatenate(vids_list, axis=0)
    if "Ch2" in recording: # Ch2 is green (calcium) channel
        tiff.imwrite(os.path.join(averaged_vid_directory, f"{n_avg}favg_{recording}.tiff"), combined_vid.astype(np.uint16))
    elif "Ch1" in recording: # Save plasma channel videos in dedicated Plasma folder
        tiff.imwrite(os.path.join(averaged_vid_directory, "Plasma", f"{n_avg}favg_{recording}.tiff"), combined_vid.astype(np.uint16))
    

'3abc'